# Rotate SH (Axis=2)

In [ ]:
import numpy as np
import torch as th
from PIL import Image
import json, glob, os

method = ["hou21_rotate_axis=2", "hou22_rotate_axis=2", "diffusionrig_rotate_axis=2", "iclight_rotate_512x512_map_centered_axis2", "relipa_rotate_axis=2", "ours_256_difareli_rotate_rot2", "ours_difareli++_oneshot_rotate_rot2_tomax"]
sample = json.load(open("/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/TPAMI_MajorRevision/rotateSH_axis2.json", "r"))
meta = json.load(open("/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/MajorRevision/figure/ffhq/rotateSH_axis2.json", "r"))
out_dir = "./rotateSH_figure/axis=2/"
os.makedirs(out_dir, exist_ok=True)

count = 0
for pid, dat in sample['pair'].items():
    src = dat['src']
    dst = dat['dst']
    if "frames" not in dat:
        continue
    frames_idx = dat['frames']

    res_img = {}
    for m in method:
        meta_dat = meta[m]
        img_dir = meta_dat['img_dir']
        itp_method = meta_dat['itp_method']
        diff_step = meta_dat['diff_step']
        n_frame_tmp = meta_dat['n_frame']

        img_list = []
        shadow_list = []
        render_list = []
        for fid in frames_idx:
            
            if m == "diffusionrig_rotate_axis=2":
                img_list.append(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/res_frame{fid:03d}.png')
            elif m == "relipa_rotate_axis=2":
                img_list.append(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/gs=4.5_ds=25/n_frames={n_frame_tmp}/256/res_frame{fid:03d}.png')
            else:
                img_list.append(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/res_frame{fid}.png')
            if m == "ours_difareli++_oneshot_rotate_rot2_tomax":
                shadow_list.append(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/dst_shadm_shad_frame{fid}.png')
                render_list.append(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/dst_ren_frame{fid}.png')

        if len(shadow_list) > 0 and len(render_list) > 0:
            shadow_list = [Image.open(f) for f in shadow_list]
            render_list = [Image.open(f) for f in render_list]

        # if len(glob.glob(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/res_frame*.png')) == 0:
        if len(img_list) == 0:
            print(f"Warning: No images found for {m} src={src} dst={dst}.")
            res_img[m] = [np.zeros((256, 256, 3)) for _ in frames_idx]
        else:
            res_img[m] = [Image.open(f) for f in img_list]
    
    # Concatenate images horizontally
    for m in method:
        res_img[m] = np.concatenate(res_img[m], axis=1)
    # Concatenate images vertically
    res_img = np.concatenate([res_img[m] for m in method], axis=0).astype(np.uint8)
    res_img = Image.fromarray(res_img)
    res_img.save(f'{out_dir}/{count+1}_{src}_{pid}.png')

    cond = []
    for cond_img in zip(render_list, shadow_list):
        # Concatenate images horizontally with render_0, shadow_0, render_1, shadow_1, ...
        cond.append(np.concatenate(cond_img, axis=1))
    cond = np.concatenate(cond, axis=1).astype(np.uint8)
    cond = Image.fromarray(cond)
    cond.save(f'{out_dir}/{count+1}_{src}_{pid}_cond.png')
    count+=1

KeyboardInterrupt: 

# Gen SH's ball for IC-light

In [48]:
import numpy as np
import torch as th
from PIL import Image
import json, glob, os
import matplotlib.pyplot as plt

axis = 1
if axis == 1:
    sample = json.load(open("./HDRI_sota_sj.json", "r"))
    ball_path = "/data/mint/DPM_Dataset/Dataset_For_Baseline/ffhq_user_study/axis=1/valid/"
    os.makedirs("./rotateSH_figure/axis=1_ball/", exist_ok=True)
elif axis == 2:
    sample = json.load(open("./selected_rotate_RT.json", "r"))
    # ball_path = "/data/mint/DPM_Dataset/Dataset_For_Baseline/ffhq_user_study/axis=2/valid/"
    ball_path = "/data/mint/DPM_Dataset/Dataset_For_Baseline/ffhq_selected_rotateSH/axis=2/valid/"
    os.makedirs("./rotateSH_figure/axis=2_ball/", exist_ok=True)

for pid, dat in sample['pair'].items():
    src = dat['src']
    dst = dat['dst']
    if "frames" not in dat:
        continue
    frames_idx = dat['frames']

    print(src, dst, frames_idx)
    if len(glob.glob(f'{ball_path}/{pid}_src={src}_dst={dst}/n_step=60/ball/m_*.png')) == 0:
        continue
    ball_img_list = [sorted(glob.glob(f'{ball_path}/{pid}_src={src}_dst={dst}/n_step=60/ball/m_*.png'))[x] for x in frames_idx]
    print(ball_img_list)

    ball_img = [Image.open(f) for f in ball_img_list]
    
    # Concatenate images horizontally
    ball_out = []
    for i in range(len(ball_img)):
        ball_out.append(np.array(ball_img[i]))
        ball_out.append(np.zeros((256, 512+256, 3)))
    ball_out = np.concatenate(ball_out, axis=1).astype(np.uint8)
    
    mask = ball_out[..., 0:1] > 0
    # plt.imshow(mask * 255)
    # plt.show()
    # assert False
    alpha_channel = (mask * 255).astype(np.uint8)

    # Combine RGB and Alpha channels
    ball_out_tp = np.concatenate((ball_out, alpha_channel), axis=2)
    
    ball_out = Image.fromarray(ball_out)
    ball_out_tp = Image.fromarray(ball_out_tp, mode='RGBA')
    if axis == 1:
        ball_out.save(f'./rotateSH_figure/axis=1_ball/{pid}.png')
        ball_out_tp.save(f'./rotateSH_figure/axis=1_ball/{pid}_tp.png')
        os.system(f'cp {ball_path}/{pid}_src={src}_dst={dst}/n_step=60/src={src} ./rotateSH_figure/axis=1_ball/{pid}_{src}.png')
    elif axis == 2:
        ball_out.save(f'./rotateSH_figure/axis=2_ball/{pid}.png')
        ball_out_tp.save(f'./rotateSH_figure/axis=2_ball/{pid}_tp.png')
        os.system(f'cp {ball_path}/{pid}_src={src}_dst={dst}/n_step=60/src={src} ./rotateSH_figure/axis=2_ball/{pid}_{src}.png')
        
    

65904.jpg 61585.jpg [10, 20, 28, 36, 52]
['/data/mint/DPM_Dataset/Dataset_For_Baseline/ffhq_user_study/axis=1/valid//pair2_src=65904.jpg_dst=61585.jpg/n_step=60/ball/m_010.png', '/data/mint/DPM_Dataset/Dataset_For_Baseline/ffhq_user_study/axis=1/valid//pair2_src=65904.jpg_dst=61585.jpg/n_step=60/ball/m_020.png', '/data/mint/DPM_Dataset/Dataset_For_Baseline/ffhq_user_study/axis=1/valid//pair2_src=65904.jpg_dst=61585.jpg/n_step=60/ball/m_028.png', '/data/mint/DPM_Dataset/Dataset_For_Baseline/ffhq_user_study/axis=1/valid//pair2_src=65904.jpg_dst=61585.jpg/n_step=60/ball/m_036.png', '/data/mint/DPM_Dataset/Dataset_For_Baseline/ffhq_user_study/axis=1/valid//pair2_src=65904.jpg_dst=61585.jpg/n_step=60/ball/m_052.png']
63311.jpg 66653.jpg [13, 25, 34, 39, 55]
['/data/mint/DPM_Dataset/Dataset_For_Baseline/ffhq_user_study/axis=1/valid//pair56_src=63311.jpg_dst=66653.jpg/n_step=60/ball/m_013.png', '/data/mint/DPM_Dataset/Dataset_For_Baseline/ffhq_user_study/axis=1/valid//pair56_src=63311.jpg_dst

In [6]:
import numpy as np
import torch as th
from PIL import Image
import json, glob, os

method = ["hou21_rotate_axis=1", "hou22_rotate_axis=1", "iclight_rotate_512x512_map_centered_axis1", "ours_difareli_rotate_rot1_0.7sh_250t", "ours_difareli++_oneshot_rotate_rot1_tomax_0.7sh"]
sample = json.load(open("./HDRI_sota_sj.json", "r"))
meta = json.load(open("/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/main_results/FFHQ_CastShadows/RotateSH_figures/ffhq_rotateSH_userstudy_axis1.json", "r"))
os.makedirs("./rotateSH_figure/axis=1/", exist_ok=True)

for pid, dat in sample['pair'].items():
    src = dat['src']
    dst = dat['dst']
    if "frames" not in dat:
        continue
    frames_idx = dat['frames']

    res_img = {}
    for m in method:
        meta_dat = meta[m]
        img_dir = meta_dat['img_dir']
        itp_method = meta_dat['itp_method']
        diff_step = meta_dat['diff_step']
        n_frame_tmp = meta_dat['n_frame']

        img_list = []
        shadow_list = []
        render_list = []
        for fid in frames_idx:
            img_list.append(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/res_frame{fid}.png')
            if m == "ours_difareli++_oneshot_rotate_rot1_tomax_0.7sh":
                shadow_list.append(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/dst_shadm_shad_frame{fid}.png')
                render_list.append(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/dst_ren_frame{fid}.png')

        if len(shadow_list) > 0 and len(render_list) > 0:
            shadow_list = [Image.open(f) for f in shadow_list]
            render_list = [Image.open(f) for f in render_list]

        if len(glob.glob(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/res_frame*.png')) == 0:
            res_img[m] = [np.zeros((256, 256, 3)) for _ in frames_idx]
        else:
            res_img[m] = [Image.open(f) for f in img_list]
    
    # Concatenate images horizontally
    for m in method:
        res_img[m] = np.concatenate(res_img[m], axis=1)
    # Concatenate images vertically
    res_img = np.concatenate([res_img[m] for m in method], axis=0).astype(np.uint8)
    res_img = Image.fromarray(res_img)
    res_img.save(f'./rotateSH_figure/axis=1/{pid}.png')

    cond = []
    for cond_img in zip(render_list, shadow_list):
        # Concatenate images horizontally with render_0, shadow_0, render_1, shadow_1, ...
        cond.append(np.concatenate(cond_img, axis=1))
    cond = np.concatenate(cond, axis=1).astype(np.uint8)
    cond = Image.fromarray(cond)
    cond.save(f'./rotateSH_figure/axis=1/{pid}_cond.png')


In [10]:
import numpy as np
import torch as th
from PIL import Image
import json, glob, os

method = ["hou21_rotate_axis=1", "hou22_rotate_axis=1", "iclight_rotate_512x512_map_centered_axis1", "ours_difareli_rotate_rot1_0.7sh_250t", "ours_difareli++_oneshot_rotate_rot1_tomax_0.6sh"]
sample = json.load(open("./HDRI_sota_sj.json", "r"))
meta = json.load(open("/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/main_results/FFHQ_CastShadows/RotateSH_figures/ffhq_rotateSH_userstudy_axis1.json", "r"))
os.makedirs("./rotateSH_figure/axis=1/", exist_ok=True)

for pid, dat in sample['pair'].items():
    src = dat['src']
    dst = dat['dst']
    if "frames" not in dat:
        continue
    frames_idx = dat['frames']

    res_img = {}
    for m in method:
        meta_dat = meta[m]
        img_dir = meta_dat['img_dir']
        itp_method = meta_dat['itp_method']
        diff_step = meta_dat['diff_step']
        n_frame_tmp = meta_dat['n_frame']

        img_list = []
        shadow_list = []
        render_list = []
        for fid in frames_idx:
            img_list.append(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/res_frame{fid}.png')
            if m == "ours_difareli++_oneshot_rotate_rot1_tomax_0.6sh":
                shadow_list.append(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/dst_shadm_shad_frame{fid}.png')
                render_list.append(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/dst_ren_frame{fid}.png')

        if len(shadow_list) > 0 and len(render_list) > 0:
            shadow_list = [Image.open(f) for f in shadow_list]
            render_list = [Image.open(f) for f in render_list]

        if len(glob.glob(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/res_frame*.png')) == 0:
            res_img[m] = [np.zeros((256, 256, 3)) for _ in frames_idx]
        else:
            res_img[m] = [Image.open(f) for f in img_list]
    
    # Concatenate images horizontally
    for m in method:
        res_img[m] = np.concatenate(res_img[m], axis=1)
    # Concatenate images vertically
    res_img = np.concatenate([res_img[m] for m in method], axis=0).astype(np.uint8)
    res_img = Image.fromarray(res_img)
    res_img.save(f'./rotateSH_figure/axis=1/{pid}.png')

    cond = []
    for cond_img in zip(render_list, shadow_list):
        # Concatenate images horizontally with render_0, shadow_0, render_1, shadow_1, ...
        cond.append(np.concatenate(cond_img, axis=1))
    cond = np.concatenate(cond, axis=1).astype(np.uint8)
    cond = Image.fromarray(cond)
    cond.save(f'./rotateSH_figure/axis=1/{pid}_cond.png')
